# Autoencoders — a hands-on MNIST notebook

*Companion to Chapter 14 of Goodfellow, Bengio & Courville — "Autoencoders."*

An autoencoder copies its input to its output **through a narrow middle that won't let it cheat**. The encoder `f` compresses `x` into a code `h`; the decoder `g` rebuilds `r = g(f(x))`. Because the copy can't be perfect, the pressure to reconstruct forces the code to keep only what matters. This notebook walks the hourglass, one runnable MNIST experiment per station.

| # | Station | What we run on MNIST |
|---|---------|----------------------|
| 1 | The blueprint | Encode→code→decode; the identity trap |
| 2 | Undercomplete | Linear AE ≈ PCA; nonlinear AE beats PCA; 2-D code map |
| 3 | Regularized | Overcomplete code copies unless a penalty shapes it |
| 4 | Sparse | L¹ on activations → few active, interpretable units |
| 5 | Denoising | Corrupt→clean; the manifold vector field |
| 6 | Contractive | Jacobian penalty → code steady off-manifold |
| 7 | Depth | Deep AE: lower error, better features per parameter |
| 8 | Stochastic / VAE | Reparameterized latent, prior samples, 2-D map |
| 9 | Encoder–decoder apps | Manifold grid, interpolation, retrieval, anomaly |

> **The one idea:** the *constraint* is the feature. A perfect copy learns nothing; a bottleneck, a sparsity/Jacobian penalty, corrupted inputs, or a probabilistic code is what forces the code to capture the data manifold.

**Runtime:** CPU works; a GPU (Runtime → Change runtime type → GPU) speeds up the training stations. Run top to bottom.


## Setup — data, an autoencoder class, and shared helpers

We use pixel values in `[0,1]` (plain `ToTensor`, no normalization) so a sigmoid decoder + reconstruction loss behaves well. One `AE` class and a few helpers power most stations.


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

transform = transforms.ToTensor()                      # pixels in [0,1]
train_full = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_full  = datasets.MNIST('./data', train=False, download=True, transform=transform)

def make_loader(n, bs=128, seed=0, train=True):
    ds = train_full if train else test_full
    idx = np.random.RandomState(seed).choice(len(ds), n, replace=False)
    return DataLoader(Subset(ds, idx), batch_size=bs, shuffle=train)

train_loader = make_loader(10000, 128)                 # AE training set
test_loader  = make_loader(4000, 256, train=False)
probe_tr = make_loader(2000, 2000, seed=5, train=True)   # for the linear-probe metric
probe_te = make_loader(2000, 2000, seed=6, train=False)
print('AE train 10000 | test 4000 | probe 2000/2000')

In [ ]:
def act_layer(name):
    return {'relu': nn.ReLU, 'tanh': nn.Tanh, 'sigmoid': nn.Sigmoid, 'identity': nn.Identity}[name]()

def mlp(dims, act='relu', last_act=None):
    layers=[]
    for i in range(len(dims)-1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims)-2:
            layers.append(act_layer(act))
        elif last_act:
            layers.append(act_layer(last_act))
    return nn.Sequential(*layers)

class AE(nn.Module):
    """enc_dims=[784,...,code]; decoder mirrors it. code_act shapes the code layer."""
    def __init__(self, enc_dims, act='relu', code_act=None, out_act='sigmoid'):
        super().__init__()
        self.encoder = mlp(enc_dims, act, last_act=code_act)
        self.decoder = mlp(enc_dims[::-1], act, last_act=out_act)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = self.encoder(x)
        return self.decoder(h), h

In [ ]:
def train_ae(model, loader, epochs=15, lr=1e-3, loss_type='bce', l1=0.0, denoise=0.0, log=False):
    model.to(device); opt = torch.optim.Adam(model.parameters(), lr)
    hist=[]
    for ep in range(epochs):
        model.train(); tot=0.0; nb=0
        for xb, _ in loader:
            x = xb.to(device).view(xb.size(0), -1)
            inp = (x + denoise*torch.randn_like(x)).clamp(0,1) if denoise>0 else x
            opt.zero_grad()
            r, h = model(inp)
            rec = F.binary_cross_entropy(r, x, reduction='mean')*784 if loss_type=='bce' else F.mse_loss(r, x)
            (rec + l1*h.abs().mean()).backward(); opt.step()
            tot += rec.item(); nb += 1
        hist.append(tot/nb)
        if log and (ep+1) % 5 == 0: print(f'  epoch {ep+1:2d} | recon {hist[-1]:.4f}')
    return hist

@torch.no_grad()
def eval_recon(model, loader):
    model.eval(); tot=0.0; n=0
    for xb,_ in loader:
        x = xb.to(device).view(xb.size(0), -1); r,_ = model(x)
        tot += F.mse_loss(r, x, reduction='sum').item(); n += x.numel()
    return tot/n

@torch.no_grad()
def get_codes(model, loader):
    model.eval(); H=[]; Y=[]
    for xb, yb in loader:
        _, h = model(xb.to(device).view(xb.size(0), -1)); H.append(h.cpu()); Y.append(yb)
    return torch.cat(H), torch.cat(Y)

def probe_acc(model, epochs=60):
    """Representation quality: train a linear classifier on FROZEN codes."""
    Htr, Ytr = get_codes(model, probe_tr); Hte, Yte = get_codes(model, probe_te)
    Htr, Ytr, Hte, Yte = [t.to(device) for t in (Htr, Ytr, Hte, Yte)]
    clf = nn.Linear(Htr.size(1), 10).to(device); opt = torch.optim.Adam(clf.parameters(), 1e-2)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(clf(Htr), Ytr).backward(); opt.step()
    return (clf(Hte).argmax(1) == Yte).float().mean().item()

def show_recon(model, n=8, noise=0.0, title=''):
    model.eval()
    xb,_ = next(iter(test_loader)); x = xb[:n].to(device).view(n, -1)
    inp = (x + noise*torch.randn_like(x)).clamp(0,1) if noise>0 else x
    with torch.no_grad(): r,_ = model(inp)
    fig, ax = plt.subplots(2, n, figsize=(n*1.2, 2.6))
    for i in range(n):
        ax[0,i].imshow(inp[i].cpu().view(28,28), cmap='gray'); ax[0,i].axis('off')
        ax[1,i].imshow(r[i].cpu().view(28,28),   cmap='gray'); ax[1,i].axis('off')
    ax[0,0].set_title('input', loc='left', fontsize=9); ax[1,0].set_title('reconstruction', loc='left', fontsize=9)
    plt.suptitle(title); plt.tight_layout(); plt.show()

## Station 1 — The autoencoder blueprint

Encode `h = f(x)`, decode `r = g(h)`, and minimize a reconstruction loss `L(x, g(f(x)))`. We never want a *perfect* copy: if the code is as wide as the input and the net is free enough, the easiest solution is the **identity map** — perfect score, zero insight. Everything in this chapter blocks that shortcut. First a normal undercomplete AE (code 32), then a demonstration of the identity trap.


In [ ]:
torch.manual_seed(0)
ae = AE([784, 128, 32])
h = train_ae(ae, train_loader, epochs=15, log=True)
print(f'test recon MSE: {eval_recon(ae, test_loader):.4f}')
show_recon(ae, title='Station 1 — a 32-number code rebuilds the digit')

In [ ]:
# The identity trap: a wide, unconstrained code can copy without learning structure.
torch.manual_seed(0)
ae_wide = AE([784, 784], out_act='sigmoid')            # code as wide as input
_ = train_ae(ae_wide, train_loader, epochs=10)
print(f'wide code (784)  test recon MSE: {eval_recon(ae_wide, test_loader):.4f}  <- near-perfect copy')
print(f'narrow code (32) test recon MSE: {eval_recon(ae,      test_loader):.4f}  <- forced to compress')
print('A perfect copy is the failure mode: it memorized a pass-through, learned no structure.')

## Station 2 — Undercomplete autoencoders (≈ PCA)

Making the code **smaller than the input** forces compression. With **linear** encoder/decoder and squared error, the optimal code spans the same subspace as **PCA** — the autoencoder rediscovers principal components. A **nonlinear** AE can fold along curved manifolds PCA can't see, so it reconstructs better at the same code size.


In [ ]:
# Linear AE (code=2, no nonlinearity) vs PCA (code=2)
torch.manual_seed(0)
lin_ae = AE([784, 2], act='identity', out_act='identity')
train_ae(lin_ae, train_loader, epochs=20, loss_type='mse')
lin_mse = eval_recon(lin_ae, test_loader)

# PCA via SVD on the training subset
Xtr = torch.stack([train_full[i][0] for i in np.random.RandomState(0).choice(len(train_full),10000,False)]).view(-1,784).numpy()
mu = Xtr.mean(0); U,S,Vt = np.linalg.svd(Xtr-mu, full_matrices=False)
Xte = torch.stack([test_full[i][0] for i in range(4000)]).view(-1,784).numpy()
def pca_mse(k):
    Z = (Xte-mu) @ Vt[:k].T; R = Z @ Vt[:k] + mu
    return ((Xte-R)**2).mean()
print(f'linear AE (code 2) test MSE : {lin_mse:.4f}')
print(f'PCA       (code 2) test MSE : {pca_mse(2):.4f}   <- essentially the same subspace')

In [ ]:
# Nonlinear AE (code=2) beats the flat PCA subspace, and gives a 2-D map of the data.
torch.manual_seed(0)
nl_ae = AE([784, 256, 64, 2], act='relu')
train_ae(nl_ae, train_loader, epochs=20, loss_type='mse')
print(f'nonlinear AE (code 2) test MSE: {eval_recon(nl_ae, test_loader):.4f}  vs PCA {pca_mse(2):.4f}')

H, Y = get_codes(nl_ae, test_loader)
plt.figure(figsize=(6.5,5.5))
sc = plt.scatter(H[:,0], H[:,1], c=Y, cmap='tab10', s=6, alpha=0.6)
plt.colorbar(sc, label='digit'); plt.title('Station 2 — 2-D nonlinear code map (colour = digit)')
plt.xlabel('code[0]'); plt.ylabel('code[1]'); plt.show()

## Station 3 — Regularized autoencoders

Shrinking the code is blunt. **Regularized** autoencoders keep a code that may be as large as the input — even **overcomplete** — but add a penalty that shapes it, so the penalty (not the size) blocks the identity shortcut. We measure representation quality with a **linear probe**: freeze the code, train a linear classifier on it, report test accuracy. A wide *unregularized* code reconstructs almost perfectly yet makes weaker features.


In [ ]:
configs = []
torch.manual_seed(0); a32 = AE([784,128,32]);   train_ae(a32, train_loader, epochs=15)
torch.manual_seed(0); aOC = AE([784,1024], code_act='relu'); train_ae(aOC, train_loader, epochs=15)
for name, m in [('undercomplete code=32', a32), ('overcomplete code=1024 (no reg)', aOC)]:
    print(f'{name:34s} | recon MSE {eval_recon(m, test_loader):.4f} | linear-probe acc {probe_acc(m):.3f}')
print('\nBigger code -> lower reconstruction error, but not necessarily better features.')
print('Station 4 adds a sparsity penalty to the wide code and recovers useful structure.')

## Station 4 — Sparse autoencoders

Add a penalty on the **code activations**: `Ω(h) = λ Σ|hᵢ|`. Reconstruction still wants the code to explain `x`, but the penalty pushes most units toward zero, so each input activates only a **few specialized units**. (This penalizes activations — a function of the data — not the weights.)


In [ ]:
# Sweep the sparsity weight: more lambda -> fewer active code units.
def frac_active(model, thresh=0.05):
    H,_ = get_codes(model, probe_te); return (H.abs() > thresh).float().mean().item()

rows=[]
for lam in [0.0, 0.001, 0.01, 0.05]:
    torch.manual_seed(0); m = AE([784, 256], code_act='relu')
    train_ae(m, train_loader, epochs=15, l1=lam)
    rows.append((lam, eval_recon(m, test_loader), frac_active(m), probe_acc(m), m))
    print(f'lambda={lam:<6} | recon {rows[-1][1]:.4f} | active units {rows[-1][2]*100:5.1f}% | probe {rows[-1][3]:.3f}')
sparse_ae = rows[2][4]   # keep the lambda=0.01 model for the feature plot

In [ ]:
# Visualize learned features (decoder weight columns) — sparse codes give stroke/part detectors.
W = sparse_ae.decoder[0].weight.detach().cpu()        # [784, code]
idx = W.abs().sum(0).argsort(descending=True)[:16]     # most-used features
plt.figure(figsize=(8,8))
for j, k in enumerate(idx):
    plt.subplot(4,4,j+1); plt.imshow(W[:,k].view(28,28), cmap='RdBu'); plt.axis('off')
plt.suptitle('Station 4 — features learned by a sparse autoencoder (stroke/part detectors)')
plt.tight_layout(); plt.show()

## Station 5 — Denoising autoencoders

A DAE changes the **task**: corrupt the input to `x̃`, ask for the **clean** `x` back. To undo corruption the model must learn where real data lives — the **data manifold**. The reconstruction `g(f(x̃)) − x̃` becomes a vector pointing corrupted inputs back toward the manifold (∝ the score `∇ₓ log p(x)`), the idea behind diffusion models.


In [ ]:
torch.manual_seed(0)
dae = AE([784, 256, 64])
train_ae(dae, train_loader, epochs=18, denoise=0.5, log=True)
print(f'linear-probe acc (denoising features): {probe_acc(dae):.3f}')
show_recon(dae, noise=0.5, title='Station 5 — top: noisy input,  bottom: denoised reconstruction')

In [ ]:
# The manifold vector field (Fig 14.4): a tiny DAE on a 1-D curve in 2-D.
t = np.linspace(-3, 3, 400); manifold = np.stack([t, np.sin(t)], 1).astype('float32')
Xm = torch.tensor(manifold)
class Toy(nn.Module):
    def __init__(s): super().__init__(); s.enc=nn.Sequential(nn.Linear(2,32),nn.Tanh(),nn.Linear(32,1)); s.dec=nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,2))
    def forward(s,x): return s.dec(s.enc(x))
toy = Toy(); opt = torch.optim.Adam(toy.parameters(), 5e-3)
for _ in range(3000):
    b = Xm[torch.randint(0, len(Xm), (128,))]
    noisy = b + 0.35*torch.randn_like(b)
    opt.zero_grad(); F.mse_loss(toy(noisy), b).backward(); opt.step()

gx, gy = np.meshgrid(np.linspace(-3.5,3.5,22), np.linspace(-2.2,2.2,16))
grid = torch.tensor(np.stack([gx.ravel(), gy.ravel()],1).astype('float32'))
with torch.no_grad(): arrows = (toy(grid) - grid).numpy()
plt.figure(figsize=(8,5))
plt.plot(manifold[:,0], manifold[:,1], color='teal', lw=2, label='data manifold')
plt.quiver(grid[:,0], grid[:,1], arrows[:,0], arrows[:,1], color='crimson', alpha=0.7, width=0.003)
plt.legend(); plt.title('Station 5 — DAE reconstruction field points back to the manifold'); plt.show()

## Station 6 — Contractive autoencoders

A CAE penalizes the **encoder's sensitivity to the input**: `Ω = λ‖∂f/∂x‖²_F`. The code is pushed to *not change* when the input wiggles — except along the manifold, where reconstruction forces it to stay responsive. For a single sigmoid encoder layer the Jacobian penalty has a clean analytic form `Σⱼ (hⱼ(1−hⱼ))² Σᵢ Wⱼᵢ²`, which we use directly.


In [ ]:
class CAE(nn.Module):
    def __init__(self, code=64):
        super().__init__(); self.enc = nn.Linear(784, code); self.dec = nn.Linear(code, 784)
    def forward(self, x):
        h = torch.sigmoid(self.enc(x)); return torch.sigmoid(self.dec(h)), h

def train_cae(lam, epochs=15):
    torch.manual_seed(0); m = CAE().to(device); opt = torch.optim.Adam(m.parameters(), 1e-3)
    W = m.enc.weight
    for ep in range(epochs):
        for xb,_ in train_loader:
            x = xb.to(device).view(xb.size(0), -1)
            opt.zero_grad(); r, h = m(x)
            rec = F.binary_cross_entropy(r, x, reduction='mean')*784
            jac = ((h*(1-h))**2 @ (W**2).sum(1)).mean()     # analytic ||df/dx||_F^2
            (rec + lam*jac).backward(); opt.step()
    return m

def sensitivity(model, eps=0.1, trials=200):
    xb,_ = next(iter(test_loader)); x = xb[:trials].to(device).view(trials,-1)
    with torch.no_grad():
        _,h0 = model(x); _,h1 = model((x+eps*torch.randn_like(x)).clamp(0,1))
    return (h1-h0).norm(dim=1).mean().item()

cae0 = train_cae(0.0); caeL = train_cae(0.1)
print(f'plain AE (lambda=0)  : code shift under input noise = {sensitivity(cae0):.3f}')
print(f'contractive (lambda=0.1): code shift under input noise = {sensitivity(caeL):.3f}  <- steadier')
print('The CAE code barely moves for off-manifold nudges — insensitive to nuisance, robust features.')

## Station 7 — Representational power, size & depth

Encoder and decoder are feedforward nets, so depth pays the same dividend: a **deep** encoder can represent mappings a shallow one needs exponentially more units for, reaching **lower reconstruction error** and better features for a comparable parameter budget. We compare a shallow and a deep AE with the same code size.


In [ ]:
torch.manual_seed(0); shallow = AE([784, 32])
torch.manual_seed(0); deep    = AE([784, 256, 128, 64, 32])
train_ae(shallow, train_loader, epochs=18); train_ae(deep, train_loader, epochs=18)
def nparams(m): return sum(p.numel() for p in m.parameters())
for name, m in [('shallow [784,32]', shallow), ('deep [784,256,128,64,32]', deep)]:
    print(f'{name:28s} | params {nparams(m):>8,} | recon MSE {eval_recon(m, test_loader):.4f} | probe {probe_acc(m):.3f}')
print('\nDepth lowers reconstruction error and improves the code — hierarchical features, not decoration.')

## Station 8 — Stochastic encoders & decoders → the VAE

Generalize encoder and decoder into **distributions**: the encoder outputs `p(h|x)` (a spread of codes), the decoder `p(x|h)`. Training minimizes `−log p(x|h)` plus a KL term to a prior — the **variational autoencoder**. The reparameterization trick `z = μ + σ·ε` keeps it differentiable. A 2-D latent lets us *see* and *sample* the space.


In [ ]:
class VAE(nn.Module):
    def __init__(self, zdim=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784,400), nn.ReLU())
        self.mu = nn.Linear(400, zdim); self.lv = nn.Linear(400, zdim)
        self.dec = nn.Sequential(nn.Linear(zdim,400), nn.ReLU(), nn.Linear(400,784), nn.Sigmoid())
    def forward(self, x):
        e = self.enc(x); mu, lv = self.mu(e), self.lv(e)
        z = mu + torch.exp(0.5*lv)*torch.randn_like(mu)     # reparameterize
        return self.dec(z), mu, lv, z

torch.manual_seed(0); vae = VAE(zdim=2).to(device); opt = torch.optim.Adam(vae.parameters(), 1e-3)
for ep in range(20):
    vae.train()
    for xb,_ in train_loader:
        x = xb.to(device).view(xb.size(0),-1); opt.zero_grad()
        r, mu, lv, _ = vae(x)
        bce = F.binary_cross_entropy(r, x, reduction='sum')/x.size(0)
        kld = -0.5*torch.sum(1+lv-mu.pow(2)-lv.exp())/x.size(0)
        (bce+kld).backward(); opt.step()
print('VAE trained (2-D latent).')

with torch.no_grad():
    samples = vae.dec(torch.randn(8,2, device=device))
plt.figure(figsize=(9,1.6))
for i in range(8):
    plt.subplot(1,8,i+1); plt.imshow(samples[i].cpu().view(28,28), cmap='gray'); plt.axis('off')
plt.suptitle('Station 8 — digits generated by sampling the prior z ~ N(0, I)'); plt.show()

In [ ]:
# 2-D latent map: encode the test set, colour by digit.
vae.eval(); Z=[]; Y=[]
with torch.no_grad():
    for xb,yb in test_loader:
        e = vae.enc(xb.to(device).view(xb.size(0),-1)); Z.append(vae.mu(e).cpu()); Y.append(yb)
Z = torch.cat(Z); Y = torch.cat(Y)
plt.figure(figsize=(6.5,5.5))
sc = plt.scatter(Z[:,0], Z[:,1], c=Y, cmap='tab10', s=6, alpha=0.6)
plt.colorbar(sc, label='digit'); plt.title('Station 8 — VAE latent space (colour = digit)')
plt.xlabel('z[0]'); plt.ylabel('z[1]'); plt.show()

## Station 9 — Encoder–decoder applications

Once you can map data to a code and back, the code is useful far beyond reconstruction: **dimensionality reduction**, **retrieval** (nearest-neighbour in code space), **denoising**, **generation**, and **interpolation**. When the two ends handle different modalities you get **sequence-to-sequence** (translation, summarization) and **U-Net**/**Transformer** encoder–decoders. We show the manifold grid, latent interpolation, code retrieval, and anomaly detection.


In [ ]:
# (a) Manifold grid: decode a grid of latent points (the classic VAE picture).
grid = 12; xs = np.linspace(-2.5, 2.5, grid)
canvas = np.zeros((28*grid, 28*grid))
vae.eval()
with torch.no_grad():
    for i, zy in enumerate(xs):
        for j, zx in enumerate(xs):
            img = vae.dec(torch.tensor([[zx, zy]], dtype=torch.float32, device=device)).cpu().view(28,28)
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img
plt.figure(figsize=(7,7)); plt.imshow(canvas, cmap='gray'); plt.axis('off')
plt.title('Station 9a — decoding a grid over the 2-D latent space'); plt.show()

In [ ]:
# (b) Latent interpolation: morph one digit into another along a straight line in z.
xb, yb = next(iter(test_loader))
a, b = xb[0:1].to(device).view(1,-1), xb[1:2].to(device).view(1,-1)
with torch.no_grad():
    za = vae.mu(vae.enc(a)); zb = vae.mu(vae.enc(b))
    steps = torch.linspace(0,1,8, device=device)
    morph = torch.cat([vae.dec((1-t)*za + t*zb) for t in steps])
plt.figure(figsize=(9,1.6))
for i in range(8):
    plt.subplot(1,8,i+1); plt.imshow(morph[i].cpu().view(28,28), cmap='gray'); plt.axis('off')
plt.suptitle(f'Station 9b — interpolating in latent space ({yb[0].item()} → {yb[1].item()})'); plt.show()

In [ ]:
# (c) Retrieval: nearest neighbours in the autoencoder code space.
H, Y = get_codes(ae, test_loader)              # reuse the station-1 AE codes
q = 0; d = (H - H[q]).pow(2).sum(1); nn_idx = d.argsort()[:6]
imgs = torch.stack([test_full[i][0] for i in range(len(Y))])
plt.figure(figsize=(9,1.8))
for j, k in enumerate(nn_idx):
    plt.subplot(1,6,j+1); plt.imshow(imgs[k].view(28,28), cmap='gray')
    plt.title('query' if j==0 else f'nn{j}', fontsize=9); plt.axis('off')
plt.suptitle('Station 9c — nearest neighbours by code distance (semantic retrieval)'); plt.show()

# (d) Anomaly detection: reconstruction error flags off-manifold inputs.
xb,_ = next(iter(test_loader)); x = xb[0:1].to(device).view(1,-1)
rot = torch.rot90(xb[0,0], 1).reshape(1,-1).to(device)       # a rotated (unusual) digit
with torch.no_grad():
    e_norm = F.mse_loss(ae(x)[0], x).item(); e_anom = F.mse_loss(ae(rot)[0], rot).item()
print(f'reconstruction error — normal digit: {e_norm:.4f} | rotated (anomaly): {e_anom:.4f}')
print('Higher error on off-manifold inputs is the basis of autoencoder anomaly detection.')

## Key takeaways

1. **Copy, but fail productively.** Encoder `f`, code `h=f(x)`, decoder `g`; a perfect copy learns nothing — the constraint is the feature.
2. **Undercomplete ≈ (nonlinear) PCA.** A code smaller than the input forces compression; linear+MSE recovers PCA, nonlinear layers fold along curved manifolds.
3. **Regularize instead of shrink.** A penalty keeps even an overcomplete code useful; size alone doesn't guarantee good features.
4. **Sparse: few, specialized units.** `λΣ|hᵢ|` yields interpretable, part-like features.
5. **Denoising learns the manifold.** Reconstruct clean from corrupted; the field `g(f(x̃))−x̃` points to the data (∝ `∇ₓ log p(x)`).
6. **Contractive: steady off-manifold.** `λ‖∂f/∂x‖²` makes the code insensitive to nuisance directions.
7. **Depth buys efficiency.** Deep encoders reach lower error and better codes per parameter.
8. **Stochastic ⇒ generative.** Codes/reconstructions as distributions give the VAE; sample the prior to generate.
9. **Encoder–decoder everywhere.** Dimensionality reduction, retrieval, denoising, generation, interpolation, and seq2seq/U-Net/Transformer.

### Try it yourself
- Retrain the sparse AE with a KL target-rate penalty (ρ≈0.05) instead of L¹ and compare the feature plots.
- Bump the VAE latent to 16 dims and measure the linear-probe accuracy of `μ` — does a richer latent help the probe even though you can't plot it?
- Feed the denoising AE a different corruption at test time (salt-and-pepper masking) and see whether manifold learning still cleans it.

*Reference: Goodfellow, Bengio & Courville, "Deep Learning," Chapter 14.*
